# Task 1 - Inconsistencies and duplicates
Ghaydaa Medhat Mohamed Amer - 58-5281 - IET

In [1]:
import pandas as pd

raw = pd.read_csv('club_signups.csv', dtype=str)
df = raw.copy()

## Part A - Inconsistencies
### 1. Look first

In [2]:
print(df.shape)
print(df.dtypes)
for col in ['faculty', 'club', 'city', 'fee_paid']:
    print(df[col].value_counts(), '\n')

(39, 9)
signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    

### 2. Faculty, club, city
First strip spaces and lower-case, then use a dictionary for what is left.

In [3]:
for col in ['faculty', 'club', 'city']:
    df[col] = df[col].str.strip().str.lower()

df['faculty'] = df['faculty'].replace({
    'media engineering and technology': 'MET',
    'information engineering and technology': 'IET',
    'engineering and materials science': 'EMS',
    'pharma': 'Pharmacy',
    'met': 'MET', 'iet': 'IET', 'ems': 'EMS',
    'pharmacy': 'Pharmacy', 'management': 'Management',
})
df['club'] = df['club'].replace({
    'chess club': 'Chess', 'chess': 'Chess',
    'debating': 'Debate', 'debate club': 'Debate', 'debate': 'Debate',
    'soccer': 'Football', 'football': 'Football',
    'robotics': 'Robotics', 'music': 'Music',
})
df['city'] = df['city'].replace({
    'alex': 'Alexandria', 'alexandria': 'Alexandria',
    'el giza': 'Giza', 'giza': 'Giza',
    'cairo': 'Cairo',
})

assert set(df['faculty']) <= {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'}
assert set(df['club']) <= {'Robotics', 'Debate', 'Music', 'Football', 'Chess'}
assert set(df['city']) <= {'Cairo', 'Giza', 'Alexandria'}

### 3. Names and emails

In [4]:
df['full_name'] = df['full_name'].str.split().str.join(' ').str.title()
df['email'] = df['email'].str.strip().str.lower()

### 4. fee_paid to True / False

In [5]:
df['fee_paid'] = df['fee_paid'].str.strip().str.lower().map({
    'yes': True, 'y': True, 'true': True, '1': True,
    'no': False, 'n': False, 'false': False, '0': False,
})
assert df['fee_paid'].notna().all()
df['fee_paid'] = df['fee_paid'].astype(bool)

### 5. signed_up_at
Two formats are mixed: `2026-09-15 13:39` (year first) and `15/09/2026 03:12`. In the slash format the first number is the day, because it goes up to 18 and there is no month 18.

In [6]:
dates = df['signed_up_at'].str.strip()
is_iso = dates.str.contains('-')
print('year-first rows:', is_iso.sum(), '| slash rows:', (~is_iso).sum())

df['signed_up_at'] = pd.concat([
    pd.to_datetime(dates[is_iso], format='%Y-%m-%d %H:%M'),
    pd.to_datetime(dates[~is_iso], format='%d/%m/%Y %H:%M'),
]).sort_index()

year-first rows: 25 | slash rows: 14


## Part B - Duplicates
### 1. Exact duplicates

In [7]:
start = len(df)
exact = df.duplicated().sum()
df = df.drop_duplicates()
print('started with:', start, '| exact duplicates removed:', exact, '| left:', len(df))

started with: 39 | exact duplicates removed: 3 | left: 36


### 2. Same student, same club, signed up again - keep the latest

In [8]:
before = len(df)
df = df.sort_values('signed_up_at').drop_duplicates(['student_id', 'club'], keep='last')
same_club = before - len(df)
print('removed:', same_club, '| left:', len(df))

removed: 4 | left: 32


### 3. Check and save

In [9]:
assert not df.duplicated(['student_id', 'club']).any()
df = df.sort_values('signup_id')
df.to_csv('club_signups_clean.csv', index=False)
df

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
17,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
14,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
10,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
7,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
11,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
23,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
1,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


## Extra: what if duplicates are removed before fixing the club spellings?

In [10]:
test = raw.drop_duplicates()
print('exact duplicates:', raw.duplicated().sum())
print('student+club duplicates found on the messy data:', test.duplicated(['student_id', 'club']).sum())
print('student+club duplicates found after cleaning:', same_club)

exact duplicates: 3
student+club duplicates found on the messy data: 2
student+club duplicates found after cleaning: 4
